In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Configurarea gestionării zgomotului cu Estimator

<Accordion>
<AccordionItem title="Versiuni de pachete">

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unor versiuni mai noi.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Există mai multe modalități de a gestiona zgomotul, de obicei prin utilizarea diferitelor tehnici de atenuare și suprimare a erorilor pentru a evita erorile înainte ca acestea să apară. Aceste tehnici cauzează de obicei un overhead de pre-procesare. Prin urmare, este important să se realizeze un echilibru între perfecționarea rezultatelor și asigurarea că jobul se finalizează într-un timp rezonabil.

Estimator suportă următoarele tehnici de gestionare a zgomotului. Consultă [Tehnici de atenuare și suprimare a erorilor](error-mitigation-and-suppression-techniques) pentru o explicație a fiecăreia. Consultă secțiunea [Setări personalizate de erori](#advanced-error) pentru instrucțiuni privind activarea acestor tehnici.

- [decuplare dinamică](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [twirling Pauli](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## Nivel de reziliență
`resilience_level` specifică cât de multă reziliență să construiești împotriva erorilor. Nivelurile mai ridicate generează rezultate mai precise, cu prețul unor timpi de procesare mai lungi. Nivelurile de reziliență pot fi folosite pentru a configura compromisul cost/acuratețe atunci când aplici gestionarea zgomotului la interogarea primitivă. Gestionarea zgomotului reduce erorile (biasul) din rezultate prin procesarea ieșirilor dintr-o colecție sau ansamblu de circuite înrudite. Gradul de reducere a erorilor depinde de metoda aplicată. Nivelul de reziliență abstractizează alegerea detaliată a metodei de gestionare a zgomotului pentru a permite utilizatorilor să raționeze despre compromisul cost/acuratețe adecvat aplicației lor.

Astfel, fiecare nivel corespunde unei metode sau unor metode cu un nivel crescând de overhead de eșantionare cuantică pentru a-ți permite să experimentezi cu diferite compromisuri timp-acuratețe. Tabelul următor îți arată ce niveluri și metode corespunzătoare sunt disponibile pentru fiecare dintre primitive.

<span id="resilience-table"></span>

| Nivel de reziliență | Descriere | Tehnică |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0 | Fără atenuare | Niciuna |
| 1 [Implicit] | Costuri minime de atenuare: Atenuarea erorilor asociate cu erorile de citire | Twirling-ul măsurătorilor [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex) |
| 2 | Costuri medii de atenuare. De obicei reduce biasul în estimatori, dar nu este garantat să fie zero-bias. | Nivelul 1 + [Extrapolare Zero Zgomot (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) și twirling de gate

> **Info:** Nivelurile de reziliență sunt în prezent în versiune beta, astfel că overhead-ul de eșantionare și calitatea soluției vor varia de la circuit la circuit. Caracteristici noi, opțiuni avansate și instrumente de management vor fi lansate continuu. Metodele specifice de gestionare a zgomotului nu sunt garantate să fie aplicate la fiecare nivel de reziliență.

### Configurarea Estimator cu niveluri de reziliență
Poți folosi nivelurile de reziliență pentru a specifica tehnicile de gestionare a zgomotului sau poți seta tehnici personalizate individual, după cum este descris în [Setări personalizate de erori](#advanced-error).

> **Note:** Orice opțiuni pe care le specifici manual în plus față de nivelul de reziliență sunt aplicate în plus față de setul de bază de opțiuni definite de nivelul de reziliență. Prin urmare, în principiu, ai putea seta nivelul de reziliență la 1, dar apoi dezactiva atenuarea măsurătorilor, deși acest lucru nu este recomandat.
> 
> De exemplu, setarea nivelului de reziliență la 0 dezactivează `zne_mitigation`, dar `estimator.options.resilience.zne_mitigation = True` suprascrie acea valoare.

### Exemplu
Codul următor activează ZNE, TREX și twirling de gate prin setarea `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Setări personalizate de gestionare a zgomotului
Poți activa și dezactiva metodele individuale de gestionare a zgomotului folosind [opțiunile Estimator](/guides/estimator-options).

> **Note:** Nu toate opțiunile funcționează împreună cu toate tipurile de circuite. Consultă [tabelul de compatibilitate a caracteristicilor](estimator-options#options-compatibility-table) pentru detalii.

### Exemplu

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>